In [6]:
# Importando bibliotecas necessárias
import pandas as pd
import numpy as np
from pathlib import Path 

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

# Definindo o caminho para o arquivo CSV
data_raw = Path('../data/raw/online_retail_II.xlsx')
assert data_raw.exists(), f"O arquivo {data_raw} não foi encontrado."
print(f"Arquivo encontrado: {data_raw}")


Arquivo encontrado: ..\data\raw\online_retail_II.xlsx


In [7]:
# Leitura das duas abas

sheets = pd.read_excel(data_raw, sheet_name=None, engine='openpyxl')
print('Abas disponiveis:', list(sheets.keys()))
for name, df in sheets.items():
    print(f' {name}:{df.shape}')

Abas disponiveis: ['Year 2009-2010', 'Year 2010-2011']
 Year 2009-2010:(525461, 8)
 Year 2010-2011:(541910, 8)


In [8]:
# Concatenando as abas em um único DataFrame

df = pd.concat(sheets.values(), ignore_index=True)
print('Shape consolidado:', df.shape)
print('Colunas', df.columns.tolist())
df.head()

Shape consolidado: (1067371, 8)
Colunas ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [9]:
# Verificando tipos de dados e valores nulos

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [11]:
# Proporção de valores nulos por coluna

null_pct = (df.isna().sum() / len(df)*100).round(2).sort_values(ascending=False)
print('% de nulls por colunas: ')
print(null_pct)

% de nulls por colunas: 
Customer ID    22.77
Description     0.41
StockCode       0.00
Invoice         0.00
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Country         0.00
dtype: float64


In [12]:
# Estatísticas descritivas para colunas numéricas

df.describe(include='all')

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,1067371.0,1067371,1062989,1.067371e+06,1067371,1.067371e+06,824364.000000,1067371
unique,53628.0,5305,5698,NaN,NaN,NaN,NaN,43
top,537434.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1350.0,5829,5918,NaN,NaN,NaN,NaN,981330
mean,NaN,NaN,NaN,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504,NaN
min,NaN,NaN,NaN,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000,NaN
50%,NaN,NaN,NaN,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000,NaN
75%,NaN,NaN,NaN,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000,NaN
max,NaN,NaN,NaN,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000,NaN


In [14]:
# Verificando cancelamentos (quantidade negativa)

df['Invoice'] = df['Invoice'].astype(str)  # Garantindo que seja string para evitar erros
cancelados = df['Invoice'].str.startswith('C') # Cancelamentos começam com 'C'
print(f'Invoie com prefixo C (cancelados): {cancelados.sum()}')
print('% do total: ', round(cancelados.mean() * 100, 2), '%')
print()
print('Invoice C com Quantity >= 0: (inconsistente)', ((cancelados) & (df['Quantity'] >= 0)).sum()) # Quantidade positiva em cancelamentos
print('Invoice C com Quantity < 0 (desolução sem cancelamento):', ((cancelados) & (df['Quantity'] < 0)).sum()) # Quantidade negativa em cancelamentos


Invoie com prefixo C (cancelados): 19494
% do total:  1.83 %

Invoice C com Quantity >= 0: (inconsistente) 1
Invoice C com Quantity < 0 (desolução sem cancelamento): 19493


In [17]:
# Verificando duplicatas

print('Duplicatas completas:', df.duplicated().sum())
print('% do total:', round(df.duplicated().mean()*100, 2), '%')

Duplicatas completas: 34335
% do total: 3.22 %


In [19]:
# Verificando StockCode suspeitos

df['StockCode'] = df['StockCode'].astype(str)
print('StockCodes únicos:', df['StockCode'].nunique())
print()
print('Top 20 StockCodes mais frequentes:')
print(df['StockCode'].value_counts().head(20))
print()
print('Distribuição do tamanho dos StockCodes:')
print(df['StockCode'].str.len().value_counts().sort_index())
print()
print('StockCodes com tamanho suspeito (<5 chars) - costumam ser ajustes operacionais, não produtos:')
suspeitos = df[df['StockCode'].str.len() < 5]['StockCode'].value_counts().head(20)
print(suspeitos)


StockCodes únicos: 5305

Top 20 StockCodes mais frequentes:
StockCode
85123A    5829
22423     4424
85099B    4216
21212     3318
20725     3259
84879     2960
47566     2768
21232     2747
22197     2549
22383     2540
20727     2529
21931     2434
22386     2347
22469     2325
22411     2297
84991     2271
22382     2251
22384     2230
21080     2224
22086     2217
Name: count, dtype: int64

Distribuição do tamanho dos StockCodes:
StockCode
1       1713
2        283
3       1446
4       2158
5     932385
6     127590
7       1393
8        127
9         74
12       202
Name: count, dtype: int64

StockCodes com tamanho suspeito (<5 chars) - costumam ser ajustes operacionais, não produtos:
StockCode
POST    2122
DOT     1446
M       1421
C2       282
D        177
S        104
PADS      19
CRUK      16
B          6
m          5
GIFT       1
C3         1
Name: count, dtype: int64


In [20]:
# Range de datas

print('Data minima:', df['InvoiceDate'].min())
print('Data máxima:', df['InvoiceDate'].max())
print('Range em dias:', (df['InvoiceDate'].max() - df['InvoiceDate'].min()).days)
                         

Data minima: 2009-12-01 07:45:00
Data máxima: 2011-12-09 12:50:00
Range em dias: 738


In [25]:
# Verificando paises 

print('Paises unicos:', df["Country"].unique())
print()
print('Top 10 paises por volme de linhas:')
print(df['Country'].value_counts().head(10))
print()
print('% do top 1:', round(df['Country'].value_counts(normalize=True).iloc[0] * 100, 2), '%')

Paises unicos: ['United Kingdom' 'France' 'USA' 'Belgium' 'Australia' 'EIRE' 'Germany'
 'Portugal' 'Japan' 'Denmark' 'Nigeria' 'Netherlands' 'Poland' 'Spain'
 'Channel Islands' 'Italy' 'Cyprus' 'Greece' 'Norway' 'Austria' 'Sweden'
 'United Arab Emirates' 'Finland' 'Switzerland' 'Unspecified' 'Malta'
 'Bahrain' 'RSA' 'Bermuda' 'Hong Kong' 'Singapore' 'Thailand' 'Israel'
 'Lithuania' 'West Indies' 'Lebanon' 'Korea' 'Brazil' 'Canada' 'Iceland'
 'Saudi Arabia' 'Czech Republic' 'European Community']

Top 10 paises por volme de linhas:
Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64

% do top 1: 91.94 %


In [26]:
# Receita sem clientes idenficados

df['Receita'] = df['Quantity'] * df['Price']
receita_total = df['Receita'].sum()
receita_sem_cliente = df[df['Customer ID'].isna()]['Receita'].sum()
print('receita total (bruta):', round(receita_total, 2))
print('Receita sem Customer ID:', round(receita_sem_cliente, 2))
print('% da rceita sem cliente identificado:', round(receita_sem_cliente / receita_total * 100, 2), '%')

receita total (bruta): 19287250.57
Receita sem Customer ID: 2638958.18
% da rceita sem cliente identificado: 13.68 %
